# Latent Watch — Pilot Training Notebook

**Entry point for Colab training runs.**

This notebook is intentionally thin — all logic lives in `src/latent_watch/`.

## What this notebook does
1. Mounts Google Drive and installs the repo
2. Sets hyperparameters inline (override the YAML configs if you want)
3. Calls into `src/latent_watch/training/` to run the selected training mode

## Prerequisites
- The dataset pipeline (Steps 1–11) has already run and outputs are saved to Google Drive at `MyDrive/latent_watch/data/processed/beavertails_risk_v1/`

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

DRIVE_ROOT   = '/content/drive/MyDrive/latent_watch'
REPO_DIR     = f'{DRIVE_ROOT}/repo'           # git clone lives here
DATA_DIR     = f'{DRIVE_ROOT}/data/processed/beavertails_risk_v1'
CKPT_DIR     = f'{DRIVE_ROOT}/checkpoints'

os.makedirs(REPO_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)


if not os.path.exists(f'{REPO_DIR}/.git'):
    !git clone https://github.com/YOUR_ORG/latent-watch.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull


!pip install -e {REPO_DIR} --quiet


sys.path.insert(0, f'{REPO_DIR}/src')

print('Setup complete.')

In [ ]:
# Load secrets from Colab Secrets (Runtime > Secrets)
from google.colab import userdata

os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')  # only for rationale gen

# Authenticate with HuggingFace for gated Llama access
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

## 1–11. Dataset Pipeline

> **Skip this section if the dataset already exists on Drive.**  
> The processed files persist across sessions. Only run these cells on first use
> or when you want to regenerate the dataset.

Each cell corresponds to one pipeline step.

In [ ]:
# ── Dataset pipeline config (overrides configs/data/beavertails_risk.yaml) ──
PIPELINE_CFG = dict(
    source       = 'PKU-Alignment/BeaverTails',
    split        = '330k_train',
    train_size   = 5000,
    val_size     = 500,
    test_size    = 1000,
    high_risk_fraction   = 0.5,
    per_category_cap     = 400,
    random_seed          = 42,
    rationale_model      = 'gpt-4o-mini',
    interim_dir          = f'{DRIVE_ROOT}/data/interim',
    output_dir           = DATA_DIR,
)

In [ ]:
# Step 1: Load BeaverTails
from latent_watch.data.load_beavertails import load_beavertails
raw_ds = load_beavertails(split=PIPELINE_CFG['split'])
print(f'Loaded {len(raw_ds):,} rows')

In [ ]:
# Step 2: Normalize
from latent_watch.data.normalize import normalize_dataset
norm_ds = normalize_dataset(raw_ds)
print('Normalized. Sample:', norm_ds[0])

In [ ]:
# Steps 3–5: Group, apply ANY-unsafe rule, aggregate harm categories
from latent_watch.data.aggregate_prompts import aggregate_prompts, save_aggregates
agg_df = aggregate_prompts(norm_ds, source=PIPELINE_CFG['source'], source_split=PIPELINE_CFG['split'])
save_aggregates(agg_df, PIPELINE_CFG['interim_dir'])
print(agg_df['label'].value_counts())

In [ ]:
# Step 6: Split by unique prompt (before sampling, to prevent leakage)
from latent_watch.data.build_splits import build_splits, save_split_assignments
splits = build_splits(
    agg_df,
    random_seed=PIPELINE_CFG['random_seed'],
)
save_split_assignments(splits, PIPELINE_CFG['interim_dir'])

In [ ]:
# Step 7: Downsample to pilot sizes
from latent_watch.data.sample_prompts import sample_all_splits
pilot_splits = sample_all_splits(
    splits,
    train_size          = PIPELINE_CFG['train_size'],
    val_size            = PIPELINE_CFG['val_size'],
    test_size           = PIPELINE_CFG['test_size'],
    per_category_cap    = PIPELINE_CFG['per_category_cap'],
    high_risk_fraction  = PIPELINE_CFG['high_risk_fraction'],
    random_seed         = PIPELINE_CFG['random_seed'],
)

In [ ]:
# Step 8: Generate reasoning traces via OpenAI
# Rationales are generated for ALL splits (needed for CoT/COCONUT val and test eval too)
import pandas as pd
from latent_watch.data.generate_rationales import generate_rationales

rationale_splits = {}
for split_name, df in pilot_splits.items():
    print(f'\n--- {split_name} ---')
    rationale_splits[split_name] = generate_rationales(
        df,
        model=PIPELINE_CFG['rationale_model'],
        resume=True,   # skip rows that already have reasoning
    )

In [ ]:
# Step 9: Validate rationales; remove bad examples
from latent_watch.data.validate_rationales import validate_rationales

validated_splits = {}
for split_name, df in rationale_splits.items():
    print(f'\n--- {split_name} ---')
    valid_df, rejected_df = validate_rationales(df, verbose=False)
    validated_splits[split_name] = valid_df
    print(f'  Kept: {len(valid_df):,} | Rejected: {len(rejected_df):,}')

In [ ]:
# Steps 10–11: Export canonical JSONL + render answer_only and cot formats
from latent_watch.data.render_training_formats import render_all

render_all(
    splits=validated_splits,
    output_dir=PIPELINE_CFG['output_dir'],
    rationale_model=PIPELINE_CFG['rationale_model'],
)
print(f'\nDataset saved to {PIPELINE_CFG["output_dir"]}')

## Training

Select the training mode below, then run the corresponding cell.
Hyperparameters set here override the YAML configs.

In [ ]:
# ── Hyperparameters (set inline; training scripts also accept a yaml path) ──
TRAINING_CFG = dict(
    model_name_or_path          = 'meta-llama/Llama-3.2-1B-Instruct',
    dataset_dir                 = DATA_DIR,           # training scripts select subdir
    output_dir                  = CKPT_DIR,
    # LoRA
    lora_r                      = 16,
    lora_alpha                  = 32,
    lora_dropout                = 0.05,
    target_modules              = ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    # Training loop
    num_train_epochs            = 3,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-4,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.05,
    weight_decay                = 0.01,
    max_seq_length              = 256,
    bf16                        = True,
    load_in_4bit                = True,
    seed                        = 42,
    logging_steps               = 50,
)

In [ ]:
# ── Answer-only baseline ───────────────────────────────────────────────────
# Uncomment to run:

# from latent_watch.training.train_answer_only import train_answer_only
# train_answer_only(
#     dataset_dir   = f"{TRAINING_CFG['dataset_dir']}/answer_only",
#     output_dir    = f"{TRAINING_CFG['output_dir']}/llama_1b_answer_only",
#     hparams       = TRAINING_CFG,
#     config_path   = None,   # set to yaml path to load from file instead
# )

In [ ]:
# ── CoT model ─────────────────────────────────────────────────────────────
# Uncomment to run:

# from latent_watch.training.train_cot import train_cot
# train_cot(
#     dataset_dir   = f"{TRAINING_CFG['dataset_dir']}/cot",
#     output_dir    = f"{TRAINING_CFG['output_dir']}/llama_1b_cot",
#     hparams       = {**TRAINING_CFG,
#                      'per_device_train_batch_size': 4,
#                      'gradient_accumulation_steps': 8,
#                      'max_seq_length': 512},
#     config_path   = None,
# )

In [ ]:
# ── COCONUT model (starts from CoT checkpoint) ────────────────────────────
# Uncomment to run:

# from latent_watch.training.train_coconut import train_coconut
# train_coconut(
#     dataset_dir   = f"{TRAINING_CFG['dataset_dir']}/cot",   # same data as CoT
#     cot_ckpt_dir  = f"{TRAINING_CFG['output_dir']}/llama_1b_cot",
#     output_dir    = f"{TRAINING_CFG['output_dir']}/llama_1b_coconut",
#     hparams       = {**TRAINING_CFG,
#                      'learning_rate': 1e-4,
#                      'per_device_train_batch_size': 4,
#                      'gradient_accumulation_steps': 8,
#                      'max_seq_length': 512,
#                      'num_coconut_stages': 3,
#                      'epochs_per_stage': 1},
#     config_path   = None,
# )